In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install beautifulsoup4 lxml requests tqdm

In [ ]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from tqdm import tqdm
import time

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Healthcare_RAG")

URL_FILE = PROJECT_DIR / "data" / "urls" / "nhs_urls.txt"
RAW_DIR = PROJECT_DIR / "data" / "raw"

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("URL File:", URL_FILE)
print("Raw Folder:", RAW_DIR)

URL File: /content/drive/MyDrive/Healthcare_RAG/data/urls/nhs_urls.txt
Raw Folder: /content/drive/MyDrive/Healthcare_RAG/data/raw


In [ ]:
# ============================================================
# STEP 1: LOAD AND PREPARE NHS URLS
# ============================================================
# This cell performs the following tasks:
# 1. Reads all URLs from the nhs_urls.txt file.
# 2. Removes empty lines.
# 3. Converts all HTTP links to HTTPS.
# 4. Removes duplicate URLs.
# 5. Displays summary statistics.
#
# Output:
#   - A clean Python list called 'urls'
#   - Total number of unique URLs
# ============================================================

# Read every line from the URL file
with open(URL_FILE, "r", encoding="utf-8") as file:
    raw_urls = file.readlines()

# Remove whitespace and blank lines
urls = [
    url.strip()
    for url in raw_urls
    if url.strip()
]

# Convert every HTTP URL into HTTPS
urls = [
    url.replace("http://", "https://")
    for url in urls
]

# Remove duplicate URLs while preserving the original order
urls = list(dict.fromkeys(urls))

# Display summary information
print("=" * 60)
print("NHS URL DATASET SUMMARY")
print("=" * 60)
print(f"Total Unique URLs : {len(urls)}")

# Display the first five URLs as a quick verification
print("\nFirst 5 URLs:")
for i, url in enumerate(urls[:5], start=1):
    print(f"{i}. {url}")

print("\nDataset successfully loaded and cleaned.")
print("=" * 60)

NHS URL DATASET SUMMARY
Total Unique URLs : 698

First 5 URLs:
1. https://www.nhs.uk/conditions/asthma/
2. https://www.nhs.uk/conditions/autosomal-dominant-polycystic-kidney-disease-adpkd/
3. https://www.nhs.uk/conditions/bird-flu/
4. https://www.nhs.uk/conditions/angelman-syndrome/
5. https://www.nhs.uk/mental-health/conditions/anorexia/

Dataset successfully loaded and cleaned.


In [ ]:
# ============================================================
# STEP 2: DOWNLOAD AND EXTRACT A SINGLE NHS ARTICLE
# ============================================================
# This cell:
# 1. Downloads one NHS webpage.
# 2. Parses the HTML.
# 3. Extracts the page title.
# 4. Extracts all paragraph text.
# 5. Displays a preview.
#
# We test with ONE page before scraping the full dataset.
# ============================================================

# Select the first URL from our dataset
test_url = urls[0]

print(f"Testing URL:\n{test_url}\n")

# Send an HTTP GET request
response = requests.get(
    test_url,
    headers={
        "User-Agent": "Mozilla/5.0"
    },
    timeout=30
)

# Stop execution if the request failed
response.raise_for_status()

# Parse the HTML document
soup = BeautifulSoup(response.text, "lxml")

# Extract page title
title = soup.find("h1")

if title:
    title = title.get_text(strip=True)
else:
    title = "Untitled"

print("=" * 60)
print("TITLE")
print("=" * 60)
print(title)

# Extract all paragraph text
paragraphs = soup.find_all("p")

article_text = "\n\n".join(
    paragraph.get_text(" ", strip=True)
    for paragraph in paragraphs
)

print("\n" + "=" * 60)
print("ARTICLE PREVIEW")
print("=" * 60)

# Display only the first 1500 characters
print(article_text[:1500])

print("\n")
print("=" * 60)
print(f"Total Characters Extracted : {len(article_text):,}")
print(f"Total Paragraphs Found     : {len(paragraphs)}")
print("=" * 60)

Testing URL:
https://www.nhs.uk/conditions/asthma/

TITLE
Asthma

ARTICLE PREVIEW
Asthma is a common condition that affects your breathing. It cannot currently be cured, but if it's well treated you should not have problems with symptoms.

The main symptoms of asthma are problems with breathing, such as:

Asthma symptoms can be mild or severe. They usually come and go. They may be worse at night and early in the morning.

An asthma attack is when you have severe symptoms and it's difficult to breathe, which can be life-threatening.

Symptoms can be triggered by different things including exercise, high levels of air pollution, cold air, or contact with something you're allergic to, such as pollen, dust, mould or animals.

Asthma usually starts in children, but it can happen at any age.

If you're having an asthma attack, sit up straight, to help your breathing. Try to stay calm.

Use your asthma reliever inhaler if you have one:

If your personal asthma action plan has a different maxi

In [ ]:
# ============================================================
# STEP 3: EXTRACT ONLY THE MAIN ARTICLE CONTENT
# ============================================================
# This cell improves the scraper by:
# 1. Locating the main content area of the NHS page.
# 2. Extracting only headings and paragraphs.
# 3. Ignoring navigation, footer, cookies, and sidebars.
# 4. Formatting the article for saving as a text file.
# ============================================================

# Try to locate the main content section
main_content = soup.find("main")

# Check whether the main section exists
if main_content is None:
    raise ValueError("Main article section not found.")

# Store extracted text here
article_lines = []

# Extract headings and paragraphs in the order they appear
for tag in main_content.find_all(["h1", "h2", "h3", "p"]):

    # Get clean text
    text = tag.get_text(" ", strip=True)

    # Ignore empty text
    if not text:
        continue

    # Format headings differently for readability
    if tag.name == "h1":
        article_lines.append(f"# {text}")

    elif tag.name == "h2":
        article_lines.append(f"\n## {text}")

    elif tag.name == "h3":
        article_lines.append(f"\n### {text}")

    else:
        article_lines.append(text)

# Combine everything into one article
clean_article = "\n\n".join(article_lines)

# Display a preview
print("=" * 70)
print("CLEAN ARTICLE PREVIEW")
print("=" * 70)

print(clean_article[:2500])

print("\n")
print("=" * 70)
print(f"Characters Extracted : {len(clean_article):,}")
print(f"Lines Extracted      : {len(article_lines):,}")
print("=" * 70)

CLEAN ARTICLE PREVIEW
# Asthma

Asthma is a common condition that affects your breathing. It cannot currently be cured, but if it's well treated you should not have problems with symptoms.


## Symptoms of asthma

The main symptoms of asthma are problems with breathing, such as:

Asthma symptoms can be mild or severe. They usually come and go. They may be worse at night and early in the morning.

An asthma attack is when you have severe symptoms and it's difficult to breathe, which can be life-threatening.

Symptoms can be triggered by different things including exercise, high levels of air pollution, cold air, or contact with something you're allergic to, such as pollen, dust, mould or animals.

Asthma usually starts in children, but it can happen at any age.


## Non-urgent advice: See a GP if:


## How to treat an asthma attack

If you're having an asthma attack, sit up straight, to help your breathing. Try to stay calm.

Use your asthma reliever inhaler if you have one:

If your pe

In [ ]:
# ============================================================
# STEP 4: SAVE THE EXTRACTED ARTICLE AS A TEXT FILE
# ======================================
======================
# This cell:
# 1. Creates a safe filename from the page title.
# 2. Saves the extracted article as a UTF-8 text file.
# 3. Stores the file inside the 'data/raw' folder.
# ============================================================

import re

# Create a safe filename from the article title
filename = title.lower()

# Replace spaces with underscores
filename = filename.replace(" ", "_")

# Remove characters that are not allowed in filenames
filename = re.sub(r'[^a-zA-Z0-9_-]', '', filename)

# Add the .txt extension
filename = f"{filename}.txt"

# Full output path
output_file = RAW_DIR / filename

# Save the article
with open(output_file, "w", encoding="utf-8") as file:
    file.write(clean_article)

# Display confirmation
print("=" * 60)
print("ARTICLE SAVED SUCCESSFULLY")
print("=" * 60)
print(f"Filename : {filename}")
print(f"Location : {output_file}")
print("=" * 60)

ARTICLE SAVED SUCCESSFULLY
Filename : asthma.txt
Location : /content/drive/MyDrive/Healthcare_RAG/data/raw/asthma.txt


In [ ]:
# ============================================================
# STEP 5: ADVANCED ARTICLE CONTENT EXTRACTION
# ============================================================
# This cell extracts:
#   - H1, H2, H3 headings
#   - Paragraphs
#   - Bullet lists
#   - Numbered lists
#
# It also:
#   - Removes duplicate text
#   - Removes blank lines
#   - Preserves the article structure
#
# This produces a high-quality text document that will later
# be used for cleaning, chunking, embedding, and retrieval.
# ============================================================

# Locate the main article section
main_content = soup.find("main")

if main_content is None:
    raise ValueError("Main content section not found.")

# Store extracted content
article_lines = []

# Track duplicate text
seen = set()

# Extract content in the order it appears
for tag in main_content.find_all(["h1", "h2", "h3", "p", "ul", "ol"]):

    # ----------------------------
    # Handle headings
    # ----------------------------
    if tag.name == "h1":

        text = tag.get_text(" ", strip=True)

        if text and text not in seen:
            article_lines.append(f"# {text}")
            seen.add(text)

    elif tag.name == "h2":

        text = tag.get_text(" ", strip=True)

        if text and text not in seen:
            article_lines.append(f"\n## {text}")
            seen.add(text)

    elif tag.name == "h3":

        text = tag.get_text(" ", strip=True)

        if text and text not in seen:
            article_lines.append(f"\n### {text}")
            seen.add(text)

    # ----------------------------
    # Handle paragraphs
    # ----------------------------
    elif tag.name == "p":

        text = tag.get_text(" ", strip=True)

        if text and text not in seen:
            article_lines.append(text)
            seen.add(text)

    # ----------------------------
    # Handle bullet and numbered lists
    # ----------------------------
    elif tag.name in ["ul", "ol"]:

        items = tag.find_all("li", recursive=False)

        for item in items:

            text = item.get_text(" ", strip=True)

            if text and text not in seen:
                article_lines.append(f"• {text}")
                seen.add(text)

# Combine everything
clean_article = "\n\n".join(article_lines)

# Preview
print("=" * 70)
print("ADVANCED ARTICLE PREVIEW")
print("=" * 70)

print(clean_article[:3500])

print("\n")
print("=" * 70)
print(f"Characters : {len(clean_article):,}")
print(f"Lines      : {len(article_lines):,}")
print("=" * 70)

ADVANCED ARTICLE PREVIEW
# Asthma

Asthma is a common condition that affects your breathing. It cannot currently be cured, but if it's well treated you should not have problems with symptoms.


## Symptoms of asthma

The main symptoms of asthma are problems with breathing, such as:

• wheezing

• coughing

• shortness of breath

• your chest feeling tight

Asthma symptoms can be mild or severe. They usually come and go. They may be worse at night and early in the morning.

An asthma attack is when you have severe symptoms and it's difficult to breathe, which can be life-threatening.

Symptoms can be triggered by different things including exercise, high levels of air pollution, cold air, or contact with something you're allergic to, such as pollen, dust, mould or animals.

Asthma usually starts in children, but it can happen at any age.


## Non-urgent advice: See a GP if:

• you or your child have asthma symptoms

• you or your child have asthma and treatments are not helping, you're 

In [ ]:
# ============================================================
# STEP 6: DOWNLOAD ALL NHS ARTICLES
# ============================================================
# This cell:
#   1. Iterates through every URL in the dataset.
#   2. Downloads the webpage.
#   3. Extracts structured article content.
#   4. Saves each article as a text file.
#   5. Continues even if some pages fail.
#   6. Displays a progress bar and summary.
# ============================================================

import re
import time

# Statistics
success_count = 0
failed_count = 0
failed_urls = []

print("=" * 70)
print("STARTING NHS ARTICLE DOWNLOAD")
print("=" * 70)

# Loop through every URL
for url in tqdm(urls):

    try:

        # ----------------------------------------------------
        # Download webpage
        # ----------------------------------------------------
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=30
        )

        response.raise_for_status()

        # ----------------------------------------------------
        # Parse HTML
        # ----------------------------------------------------
        soup = BeautifulSoup(response.text, "lxml")

        # Locate main content
        main_content = soup.find("main")

        if main_content is None:
            raise Exception("Main article not found.")

        # ----------------------------------------------------
        # Extract title
        # ----------------------------------------------------
        title_tag = main_content.find("h1")

        if title_tag:
            title = title_tag.get_text(strip=True)
        else:
            title = "Untitled"

        # ----------------------------------------------------
        # Create safe filename
        # ----------------------------------------------------
        filename = title.lower()
        filename = filename.replace(" ", "_")
        filename = re.sub(r"[^a-zA-Z0-9_-]", "", filename)
        filename += ".txt"

        output_path = RAW_DIR / filename

        # ----------------------------------------------------
        # Extract article
        # ----------------------------------------------------
        article_lines = []
        seen = set()

        for tag in main_content.find_all(
            ["h1", "h2", "h3", "p", "ul", "ol"]
        ):

            if tag.name in ["h1", "h2", "h3"]:

                text = tag.get_text(" ", strip=True)

                if not text or text in seen:
                    continue

                seen.add(text)

                if tag.name == "h1":
                    article_lines.append(f"# {text}")

                elif tag.name == "h2":
                    article_lines.append(f"\n## {text}")

                else:
                    article_lines.append(f"\n### {text}")

            elif tag.name == "p":

                text = tag.get_text(" ", strip=True)

                if text and text not in seen:
                    article_lines.append(text)
                    seen.add(text)

            elif tag.name in ["ul", "ol"]:

                items = tag.find_all("li", recursive=False)

                for item in items:

                    text = item.get_text(" ", strip=True)

                    if text and text not in seen:
                        article_lines.append(f"• {text}")
                        seen.add(text)

        # ----------------------------------------------------
        # Save article
        # ----------------------------------------------------
        article = "\n\n".join(article_lines)

        with open(output_path, "w", encoding="utf-8") as file:
            file.write(article)

        success_count += 1

        # Be polite to the NHS website
        time.sleep(0.5)

    except Exception:

        failed_count += 1
        failed_urls.append(url)

# ============================================================
# Final Summary
# ============================================================

print("\n")
print("=" * 70)
print("SCRAPING COMPLETED")
print("=" * 70)

print(f"Articles Saved : {success_count}")
print(f"Failed Pages   : {failed_count}")

if failed_urls:
    print("\nFailed URLs:")
    for url in failed_urls:
        print(url)

print("=" * 70)

STARTING NHS ARTICLE DOWNLOAD


100%|██████████| 698/698 [14:00<00:00,  1.20s/it]



SCRAPING COMPLETED
Articles Saved : 698
Failed Pages   : 0
